# 4. Experiments

## 4.1 Performance and Stability

### 4.1.1 Setup

In [ ]:
from experiments.__init__ import create_config, figures
import matplotlib.pyplot as plt

config = create_config(system="burgers", seed=42)
cache_dir = f"./cache_seed{config.seed}"
figures.save_config_table(config)
figures.show_spec(config, keys=figures.DEFAULT_CONFIG_KEYS, title="Burgers configuration")
fig = figures.plot_3d(config=config, controller=config.ocp.LQR)
plt.show()

### 4.1.2 Data Generation

In [ ]:
from IPython.display import display
from data import load_or_generate

n_trajectories = 500
n_candidates = 8
n_MC = 500

data_train, data_val, meta = load_or_generate(config, n_trajectories=n_trajectories, controller=config.ocp.LQR)
data = data_train

summary_df = figures.save_data_summary_table(config, data)

display(summary_df.style.hide(axis="index"))

### 4.1.3 Training

In [ ]:
from controls.train import TrainConfig
from controls.model_factory import train_controllers

controller_configs = {
    "GradQRNet": {"enabled": True},
    "GradQRNet (sup)": {"enabled": True},
    "GradQRNet (pre)": {"enabled": True},
    "GradQRNet (ad)": {"enabled": True},
}

cfg = TrainConfig(batch_size=16*int(config.n_states), n_candidates=n_candidates, horizon=1, rollouts=100, lambda_sup_base=1)

figures.save_params_table(cfg, "traincfg.tex", title="Training Configuration", config=config)

trained_models, trained_controllers, histories = train_controllers(
    config=config,
    train_cfg=cfg,
    data=data_train,
    val_data=data_val,
    controller_configs=controller_configs,
)

figures.plot_training_losses(
    config=config,
    series=[(name, hist) for name, hist in histories.items()],
    logy=True,
)
plt.show()

figures.plot_training_val_mse(
    config=config,
    histories=histories,
    controller_configs=controller_configs,
)
plt.show()

figures.plot_training_losses(
    config=config,
    series=[(name, hist) for name, hist in histories.items()],
    value_key="mean_dt",
    logy=False,
    ylabel=r"mean step size $\langle \Delta t \rangle$",
    plot_name="rollout_mean_dt",
    smooth="ema",
)
plt.show()

### 4.1.4 Results

In [ ]:
from simulation import monte_carlo

controllers = [("LQR (Baseline)", config.ocp.LQR),] + [(name, trained_controllers[name]) for name in trained_controllers.keys()]

results = monte_carlo(config.ocp, config, controllers, n_MC=n_MC)
# Reuse these ICs for the combined-controller MC block below (same realizations).
burgers_mc_X0_pool = next(iter(results.values()))["X0_pool"]
figures.show_monte_carlo_results(results)
figures.save_monte_carlo_results(results, config=config, savepath="monte_carlo.tex")

In [ ]:
figures.plot_value_analysis_combined(
    config=config,
    controllers=controllers,
    n=200,
    tspan=(0.0, 2.0),
    Nt=400,
)
plt.show()

In [ ]:
controller_configs = {
    "GradQRNet (sup/ad)": {"enabled": True}, # 512 candidates
    "GradQRNet (pre/sup)": {"enabled": True},
    "GradQRNet (pre/ad)": {"enabled": True},
    "GradQRNet (pre/sup/ad)": {"enabled": True},
}

In [ ]:
trained_models, trained_controllers, histories = train_controllers(
    config=config,
    train_cfg=cfg,
    data=data_train,  
    val_data=data_val,
    controller_configs=controller_configs,  # Use the config defined above
)

In [ ]:
controllers = [(name, trained_controllers[name]) for name in trained_controllers.keys()]

results = monte_carlo(
    config.ocp, config, controllers, n_MC=n_MC, X0_pool=burgers_mc_X0_pool
)
figures.show_monte_carlo_results(results)
figures.save_monte_carlo_results(results, config=config, savepath="monte_carlo_combined.tex")

## 4.2 Generalization and Robustness

### 4.2.1 Setup

In [ ]:
config = create_config(system="allen_cahn", t1_max=30)
figures.save_config_table(config)
figures.show_spec(config, keys=figures.DEFAULT_CONFIG_KEYS, title="Allen-Cahn configuration")

fig = figures.plot_3d(config=config, controller=config.ocp.LQR)
plt.show()

### 4.2.2 Data Generation

In [ ]:
from data import load_or_generate

n_trajectories = 50
n_candidates = 64 # good 
n_MC = 250



data_ac_train, data_ac_val, meta_ac = load_or_generate(
    config, n_trajectories=n_trajectories, controller=config.ocp.LQR)

summary_ac = figures.save_data_summary_table(config, data_ac_train)
display(summary_ac.style.hide(axis="index"))

### 4.2.3 Training

In [ ]:
controller_configs_ac = {
    "GradQRNet": {"enabled": True},
    "GradQRNet (pre/sup)": {"enabled": True},
    "GradQRNet (ad)": {"enabled": True},
    "GradQRNet (sup/ad)": {"enabled": True},
}

cfg_ac = TrainConfig(batch_size=16*int(config.n_states), rollouts=25, horizon=20, n_candidates=n_candidates, dt_min=5e-4, lambda_sup_base=10, unsup_lr=2e-4)

figures.save_params_table(cfg_ac, "allen_cahn_traincfg.tex", title="Training configuration", config=config)

figures.show_spec(
    cfg_ac,
    keys=["sup_epochs", "sup_lr", "rollouts", "unsup_lr", "horizon", "batch_size"], 
    title="Training Configuration"
)


trained_models_ac, trained_controllers_ac, histories_ac = train_controllers(
    config=config,
    train_cfg=cfg_ac,
    data=data_ac_train,
    val_data=data_ac_val,
    controller_configs=controller_configs_ac,
)

figures.plot_training_losses(
    config=config,
    series=[(name, hist) for name, hist in histories_ac.items()],
    value_key="mean_dt",
    logy=False,
    ylabel=r"mean step size $\langle \Delta t \rangle$",
    plot_name="rollout_mean_dt",
    smooth="ema",
)
plt.show()



### 4.2.4 Results

In [ ]:
controllers = [(name, trained_controllers_ac[name]) for name in trained_controllers_ac.keys()]

results = monte_carlo(config.ocp, config, controllers, n_MC=n_MC)
figures.show_monte_carlo_results(results)
figures.save_monte_carlo_results(results, config=config, savepath="monte_carlo.tex")

In [ ]:
results = monte_carlo(config.ocp, config, controllers, n_MC=n_MC, dist=2.5, K=18)
figures.show_monte_carlo_results(results)
figures.save_monte_carlo_results(results, config=config, savepath="monte_carlo_far.tex")

In [ ]:
from simulation import monte_carlo_nu_mismatch

nu_baseline = config.ocp.nu
nu_eval = 0.09  # reduced diffusion
print(f"Model mismatch: train ν={nu_baseline}, eval ν={nu_eval}")
results = monte_carlo_nu_mismatch(
    config.ocp, config, controllers, nu_eval=nu_eval, n_MC=n_MC, verbose=0
)
figures.show_monte_carlo_results(results)
figures.save_monte_carlo_results(
    results, config=config, savepath="monte_carlo_nu_mismatch.tex"
)


In [ ]:
sigma_sde = 0.0015  # noise strength
dt_sde = 0.5  # Euler-Maruyama step

print(f"SDE Monte Carlo: σ={sigma_sde}, dt={dt_sde}, n_MC={n_MC}")
results = monte_carlo(
    config.ocp, config, controllers,
    n_MC=n_MC,
    sigma=sigma_sde, dt_sde=dt_sde, random_seed=42, verbose=1
)
figures.show_monte_carlo_results(results)
figures.save_monte_carlo_results(
    results, config=config, savepath="monte_carlo_sde.tex"
)


## 4.3 Scalability to Data-Scarce Regimes

### 4.3.1 Setup

In [ ]:
config = create_config(system="kuramoto_sivashinsky", nu=0.255, ic_scale=2, t1_initial=5, t1_max=20, ic_modes=3)

cache_dir = f"./cache_seed{config.seed}"
figures.save_config_table(config)
figures.show_spec(config, keys=figures.DEFAULT_CONFIG_KEYS, title="Kuramoto-Sivashinsky configuration")
fig = figures.plot_3d(config=config, controller=config.ocp.LQR)
plt.show()

In [ ]:
from simulation import monte_carlo
controllers = [("LQR (Baseline)", config.ocp.LQR)]# + [(name, trained_controllers[name]) for name in trained_controllers.keys()]
results = monte_carlo(config.ocp, config, controllers, n_MC=100)
figures.show_monte_carlo_results(results)
figures.save_monte_carlo_results(results, config=config, savepath="monte_carlo.tex")

### 4.3.3 Training

In [ ]:
controller_configs = {
    "GradQRNet": {"enabled": True},
    "GradQRNet (ad8)": {"enabled": True, "n_candidates": 8},
    "GradQRNet (ad32)": {"enabled": True, "n_candidates": 32},
    "GradQRNet (ad128)": {"enabled": True, "n_candidates": 128},
    "GradQRNet (ad512)": {"enabled": True, "n_candidates": 512},
    "GradQRNet (ad2048)": {"enabled": True, "n_candidates": 2048},
}

cfg_ks = TrainConfig(batch_size=16*int(config.n_states), n_candidates=n_candidates, horizon=20, dt_min=1e-7, rollouts=100, unsup_lr=2e-4, lambda_sup_base=1e3)


figures.save_params_table(
    cfg_ks,
    "traincfg.tex",
    title="Training Configuration",
    config=config,
    train_n_cand_tex=r"$\{1, 8, 32, 128, 512, 2048\}$",
)

figures.show_spec(
    cfg_ks,
    keys=["sup_epochs", "sup_lr", "unsup_epochs", "unsup_n_steps", "unsup_lr", "horizon", "batch_size", "n_candidates", "dt_min", "dt_max"],
    title="Training configuration"
)

trained_models, trained_controllers, histories = train_controllers(
    config=config,
    train_cfg=cfg_ks,
    controller_configs=controller_configs,
)

figures.plot_training_losses(
    config=config,
    series=[(name, hist) for name, hist in histories.items()],
    logy=True,
    smooth="ema",
    ema_alpha=0.1,
)
plt.show()

figures.plot_training_losses(
    config=config,
    series=[(name, hist) for name, hist in histories.items()],
    value_key="mean_dt",
    logy=False,
    ylabel=r"mean step size $\langle \Delta t \rangle$",
    plot_name="rollout_mean_dt",
    smooth="ema",
)
plt.show()



### 4.3.4 Results

In [ ]:
controllers = [("LQR (Baseline)", config.ocp.LQR)] + [(name, trained_controllers[name]) for name in trained_controllers.keys()]
results = monte_carlo(config.ocp, config, controllers, n_MC=n_MC)
figures.show_monte_carlo_results(results)
figures.save_monte_carlo_results(results, config=config, savepath="monte_carlo.tex")